# iSCORS — Classical Deliverable (γ + apparent α + density), no ML

The honest ACF-line output, computed by a **GPU classical fit in seconds — no network, no
training, no checkpoint** (the U-Net was retired; see `RETROSPECTIVE.md`).

- **γ map** — diffusion-rate map.
- **apparent global α** — one cell-wide anomalous exponent (~0.6); a *temporal-decorrelation*
  exponent (STICS could not verify it as true sub-diffusion — sub-PSF).
- **density G(0)=CV²** — the amplitude channel iSCORS normalises away: high-SNR, the
  cleanest single map (≈ condensation map), 'how much/many' complementary to γ's 'how fast'.

Paths/preprocessing mirror `iscors_real_runner.ipynb`. A Gradio front-end is at the end.


In [ ]:
# ── setup ────────────────────────────────────────────────────────────────
import os, subprocess, sys
REPO='https://github.com/breezy90126/iscors-net.git'; BRANCH='claude/brave-ramanujan-33eps3'
REPO_DIR='/content/iscors-net'
try:
    from google.colab import drive; drive.mount('/content/drive', force_remount=False)
except Exception: pass
if os.path.isdir(REPO_DIR):
    for c in (['git','-C',REPO_DIR,'fetch','origin'],['git','-C',REPO_DIR,'checkout',BRANCH],
              ['git','-C',REPO_DIR,'pull','origin',BRANCH]): subprocess.run(c, check=False)
else:
    subprocess.run(['git','clone','--branch',BRANCH,REPO,REPO_DIR], check=False)
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
subprocess.run(['pip','install','-q','tifffile','scipy','gradio'], check=False)
print('setup done:', os.getcwd())


In [ ]:
# ── config (same paths as iscors_real_runner.ipynb) ──────────────────────
import numpy as np
ZIP_PATH    = '/content/drive/MyDrive/iscors_test/large_file.zip'   # .zip or .tif
EXTRACT_DIR = '/content/real_data'                                  # zip extraction cache
VIDEO_FNAME = 'COBRI_rarw_video.tif'                                # name inside the zip
N_FRAMES    = 2000
BIN_FACTOR  = 2
RECON_TAUS  = (1, 2, 4, 8, 16, 32, 48, 64, 96, 128)
GAMMA_SCALE = 2.0
print('config ready')


In [ ]:
# ── reusable pipeline: load → preprocess → classical analyze ─────────────
import os, sys, numpy as np, tifffile, zipfile, tempfile
from scipy.ndimage import gaussian_filter
# robust path: don't rely on the setup cell's sys.path persisting
REPO_DIR = '/content/iscors-net'
if os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR)
    if REPO_DIR not in sys.path: sys.path.insert(0, REPO_DIR)
import importlib, utils.gpu_iscors_fit as _gf; importlib.reload(_gf)
from utils.gpu_iscors_fit import gpu_fit_maps, compute_density

def load_video(path, fname=None, n_frames=2000, bin_factor=2):
    if str(path).lower().endswith('.zip'):
        ed = globals().get('EXTRACT_DIR', '/content/real_data'); os.makedirs(ed, exist_ok=True)
        def _find(root, name):
            for dp, _, fs in os.walk(root):
                if name in fs: return os.path.join(dp, name)
            return None
        vp = _find(ed, fname)            # reuse cached extraction (no re-extract per call)
        if not vp:
            with zipfile.ZipFile(path) as z: z.extractall(ed)
            vp = _find(ed, fname)
        assert vp, f'{fname} not found in zip'
    else:
        vp = path
    H0, W0 = tifffile.imread(vp, key=0).shape
    Hb, Wb = (H0//bin_factor)*1, (W0//bin_factor)*1
    with tifffile.TiffFile(vp) as tf:
        try:    total = int(tf.series[0].shape[0])   # robust for large/BigTIFF
        except Exception: total = len(tf.pages)
    n = min(n_frames, total)
    # crop to a bin-divisible size (avoids reshape misalignment)
    Hc, Wc = Hb*bin_factor, Wb*bin_factor
    raw = np.empty((n, Hb, Wb), np.float32)
    for s in range(0, n, 100):
        e = min(s+100, n); ch = tifffile.imread(vp, key=range(s, e)).astype(np.float32)
        raw[s:e] = ch[:, :Hc, :Wc].reshape(e-s, Hb, bin_factor, Wb, bin_factor).mean((2, 4))
    # ── sanity: a >4GB classic TIFF (offset overflow) or truncated extraction
    #    yields garbage/constant frames → CV≈0 → 'no cell pixels'. Surface it here.
    finite = np.isfinite(raw).all()
    mean_I = raw.mean(0); cvm = raw.std(0) / (np.abs(mean_I) + 1e-10)
    ncell = int((cvm >= 0.005).sum())
    print(f'[load] {os.path.basename(vp)}: {n}/{total} frames  raw {raw.shape}  '
          f'mean={raw.mean():.3g} std={raw.std():.3g} min={raw.min():.3g} max={raw.max():.3g}')
    print(f'[load] cell pixels @CV>=0.005: {ncell}/{Hb*Wb}  ({100*ncell/(Hb*Wb):.1f}%)')
    if (not finite) or raw.std() < 1e-9 or ncell == 0:
        print('[load] *** DEGENERATE DATA *** — almost certainly a TIFF read problem:')
        print('       - >4GB classic TIFF: page offsets overflow at ~4GB (your warning was'
              ' offset≈4.10GB) → frames near/after 4GB read as garbage. Lower N_FRAMES so the'
              ' read stays under 4GB, or re-save the video as BigTIFF.')
        print('       - truncated extraction (disk full): re-extract, check `df -h /content`.')
        print('       Compare with iscors_real_runner.ipynb (same loader) to confirm.')
    return raw

def preprocess(raw):
    ff = raw / (np.median(raw, axis=0)[None] + 1e-10)           # flat-field
    proc = np.empty_like(ff)
    for t in range(len(ff)):
        proc[t] = ff[t] / (gaussian_filter(ff[t], sigma=4) + 1e-10)  # per-frame BG removal
    return proc

def analyze(video_proc):
    out = gpu_fit_maps(video_proc, recon_taus=RECON_TAUS, n_components=1, global_alpha=True,
                       gamma_scale=GAMMA_SCALE, min_cv=0.005, n_steps=500, verbose=False)
    cell = out['cell_mask']
    dens, _ = compute_density(video_proc, min_cv=0.005)
    return dict(gamma=out['gamma'], alpha_global=float(np.nanmedian(out['alpha'][cell])),
                density=dens, cell=cell)

def make_fig(data, cmap, title):
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(5, 4.5))
    p1, p99 = np.nanpercentile(data, 1), np.nanpercentile(data, 99)
    im = ax.imshow(data, cmap=cmap, vmin=p1, vmax=p99); plt.colorbar(im, ax=ax)
    ax.set_title(title, fontsize=11); ax.axis('off'); fig.tight_layout(); return fig
print('pipeline defined')


In [ ]:
# ── run on the configured file + plot ────────────────────────────────────
import matplotlib.pyplot as plt, time
t0 = time.time()
raw = load_video(ZIP_PATH, VIDEO_FNAME, N_FRAMES, BIN_FACTOR)
video_proc = preprocess(raw)
res = analyze(video_proc)
print(f'done in {time.time()-t0:.1f}s   apparent global α = {res["alpha_global"]:.3f}   '
      f'cell={100*res["cell"].mean():.1f}%')
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
for a_, d, cm, ttl in [(ax[0], res['gamma'], 'magma', 'γ (diffusion rate)'),
                       (ax[1], res['density'], 'viridis', 'density  G(0)=CV²')]:
    p1, p99 = np.nanpercentile(d, 1), np.nanpercentile(d, 99)
    im = a_.imshow(d, cmap=cm, vmin=p1, vmax=p99); plt.colorbar(im, ax=a_)
    a_.set_title(ttl, fontsize=11); a_.axis('off')
plt.suptitle(f'iSCORS classical deliverable — apparent global α ≈ {res["alpha_global"]:.2f}', fontsize=13)
plt.tight_layout(); plt.show()


In [ ]:
# ── Gradio front-end (layout preview) ────────────────────────────────────
import gradio as gr
def _run(file, bin_factor, n_frames):
    if file is None: raise gr.Error('請上傳 .tif（或含 tif 的 .zip）')
    fname = VIDEO_FNAME if str(file).lower().endswith('.zip') else None
    raw = load_video(file, fname, int(n_frames), int(bin_factor))
    res = analyze(preprocess(raw))
    stats = (f'cell pixels      : {100*res["cell"].mean():.1f}%\n'
             f'apparent global α: {res["alpha_global"]:.3f}  (α=1 ⇒ normal)\n'
             f'γ  median(cell)  : {np.nanmedian(res["gamma"][res["cell"]]):.3f}\n'
             f'density median   : {np.nanmedian(res["density"][res["cell"]]):.4f}  (CV²)')
    return make_fig(res['gamma'],'magma','γ (diffusion rate)'), \
           make_fig(res['density'],'viridis','density  G(0)=CV²'), stats

with gr.Blocks(title='iSCORS classical (γ + α + density)') as demo:
    gr.Markdown('## iSCORS classical deliverable — GPU fit, no ML\n'
                'γ map + apparent global α + density G(0)=CV². Seconds, no training, no checkpoint.')
    with gr.Row():
        with gr.Column(scale=1):
            f_in = gr.File(label='影片 (.tif 或含 tif 的 .zip)', file_types=['.tif','.tiff','.zip'], type='filepath')
            b_in = gr.Slider(label='BIN_FACTOR', minimum=1, maximum=8, step=1, value=2)
            n_in = gr.Number(label='N_FRAMES', value=2000, precision=0)
            run  = gr.Button('分析', variant='primary')
        with gr.Column(scale=2):
            with gr.Tabs():
                with gr.Tab('γ 擴散速率'):    g_out = gr.Plot()
                with gr.Tab('密度 G(0)=CV²'): d_out = gr.Plot()
            s_out = gr.Textbox(label='統計摘要', lines=5)
    run.click(_run, inputs=[f_in, b_in, n_in], outputs=[g_out, d_out, s_out])

demo.launch(share=True)
